# Quick start: compare all formulations

This notebook generates a two-dimensional instance, builds one common warm start, solves the compact big-M, pairwise, and MPCO methods, and compares solution quality, runtime, and convergence.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from sdp_clusters import (Data, WarmStartOptions, build_warm_start,
                          plot_comparison, plot_history, solve_compact,
                          solve_pairwise, solve_mpco)

In [ ]:
TAU = 2  # choose 1, 1.5, 2, 3, or np.inf
data = Data.blobs(n=20, facilities=3, spread=0.75, seed=7, tau=TAU)
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(data.points[:, 0], data.points[:, 1], color='black', s=35)
ax.set_title(f'{data.name}: demand points')
ax.set_aspect('equal', adjustable='datalim')
plt.show()

## Common warm start

Change `strategy` among `none`, `multistart`, `alternating`, `sdp`, and `best`. The same resulting incumbent is passed to every exact method.

In [ ]:
warm_options = WarmStartOptions(
    strategy='best', random_starts=8, alternating_iterations=40,
    sdp_time_limit=30, seed=7, log_output=False
)
warm = build_warm_start(data, warm_options)
warm.summary()

In [ ]:
TIME_LIMIT = 120
results = {
    'compact': solve_compact(data, time_limit=TIME_LIMIT, warm_start=warm),
    'pairwise': solve_pairwise(data, time_limit=TIME_LIMIT, warm_start=warm),
    'mpco': solve_mpco(data, time_limit=TIME_LIMIT, warm_start=warm),
}
pd.DataFrame([result.summary() for result in results.values()]).set_index('method')

In [ ]:
plot_comparison(data, results)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_history(results['pairwise'], ax=axes[0])
plot_history(results['mpco'], ax=axes[1])
fig.tight_layout()
plt.show()

A time-limited result remains useful: the returned objective is the best feasible solution found, while `lower_bound` and `gap` quantify the remaining proof effort.